# PydanticAI 멀티모달 종합 인사이트 도출

## 목표
- `outputs/coupang_product_comparison/` 폴더의 비교 분석 텍스트 + 시각화 이미지를 종합 분석합니다.
- PydanticAI의 **멀티모달 입력** (텍스트 + 이미지)으로 실행 가능한 비즈니스 인사이트를 도출합니다.

| 학습 목표 | 내용 |
|---|---|
| 멀티모달 입력 | `BinaryContent`로 이미지를 Agent에 전달 |
| 복잡한 중첩 스키마 | 6개 서브 스키마를 조합한 ComprehensiveInsight |
| 비즈니스 인사이트 | 순위, 측면별 분석, 감정, 트렌드, 개선 제안 |

**선수 학습**: 실전_1_2 (제품 비교 리뷰 분석), 기본_4 (이미지 이해)

### 환경 설정

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import json
from pathlib import Path
from typing import List, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models.google import GoogleModelSettings

# .env 파일에서 API 키 로드
load_dotenv()

# 멀티모달 + 대형 구조화 출력 작업이므로 더 강력한 모델을 사용합니다
# flash-lite는 복잡한 스키마를 안정적으로 생성하기 어려울 수 있습니다
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
model_id = f'google-gla:{gemini_model}'

# 데이터 경로 설정
ANALYSIS_DIR = Path('outputs/coupang_product_comparison')

# API 키 검증
api_key = os.getenv('GEMINI_API_KEY')
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'O' if api_key_valid else 'X'}")
print(f"모델: {model_id}")
print(f"데이터 경로: {ANALYSIS_DIR}")

### 1. 인사이트 스키마 정의

**스키마 설계 프로세스**

| 설계 단계 | 내용 |
|---|---|
| 1. 비즈니스 질문 | 3개 제품 중 어떤 것이 가장 우수하고, 각 제품은 어떻게 개선해야 하는가? |
| 2. 필요한 정보 | 순위, 측면별 인사이트, 감정 분석, 트렌드, 개선 제안, 시장 분석 |
| 3. 스키마 구조 | 6개 서브 스키마 => `ComprehensiveInsight` 조합 |
| 4. 제약조건 | `Literal`로 측면/우선순위 고정, `max_length`로 텍스트 제한 |

**기본_3/실전_1_1과의 연결**: `AspectInsight.aspect`는 실전_1_1의 `AspectAnalysis.aspect`와 동일한 10가지 항목을 사용합니다.

**실습 (1)**
- 6개 서브 스키마와 최상위 `ComprehensiveInsight` 스키마를 정의합니다.
- 각 서브 스키마는 하나의 분석 관점을 담당합니다: 순위, 측면별 분석, 감정, 트렌드, 개선 제안.

In [ ]:
# =============================================================================
#  종합 인사이트 스키마 정의
# =============================================================================

# 시각화 차트 해석
class ChartInterpretation(BaseModel):
    """개별 시각화 차트에 대한 해석"""
    chart_name: str = Field(description="차트 이름 (예: 평균 평점 및 추천도 비교)")
    data_observation: str = Field(
        description="차트에서 관찰된 데이터 패턴 (수치, 분포, 비율 등 구체적으로)",
        max_length=300
    )
    business_implication: str = Field(
        description="이 패턴이 소비자/비즈니스에 시사하는 점",
        max_length=200
    )


# 제품별 프로필 
class ProductRanking(BaseModel):
    """제품별 종합 순위 및 평가"""
    rank: int = Field(description="종합 순위 (1위부터)")
    product_name: str = Field(description="제품명")
    overall_score: str = Field(
        description="종합 점수 요약 (예: 평점 4.96, 추천도 0.93)",
        max_length=100
    )
    strengths: List[str] = Field(
        description="주요 강점 (차트 데이터 근거 포함, 예: 사용성 긍정비율 98%)",
        min_length=2, max_length=5
    )
    weaknesses: List[str] = Field(
        description="주요 약점 (차트 데이터 근거 포함)",
        min_length=1, max_length=5
    )
    key_keywords: List[str] = Field(
        description="워드클라우드에서 확인된 핵심 키워드 5~8개",
        min_length=3, max_length=8
    )
    recommended_for: str = Field(
        description="추천 대상 (피부타입, 용도, 예산 등 구체적으로)",
        max_length=150
    )
    not_recommended_for: str = Field(
        description="비추천 대상 (어떤 사용자에게 부적합한지)",
        max_length=150
    )


# 측면별 인사이트
class AspectInsight(BaseModel):
    """측면별 상세 인사이트"""
    aspect: Literal[
        "품질", "사용성", "기능성", "디자인", "가성비",
        "내구성", "고객서비스", "배송", "포장", "전반적"
    ] = Field(description="분석 측면")
    best_product: str = Field(description="해당 측면에서 가장 우수한 제품")
    worst_product: str = Field(description="해당 측면에서 가장 부족한 제품")
    positive_ratio_comparison: str = Field(
        description="제품 간 긍정 비율 비교 (히트맵/레이더 차트 기반, 예: A 95% vs B 78%)",
        max_length=150
    )
    key_finding: str = Field(
        description="해당 측면에 대한 핵심 발견사항",
        max_length=200
    )


# 감정 분석
class SentimentInsight(BaseModel):
    """감정 분석 인사이트"""
    most_positive_product: str = Field(description="가장 긍정적인 반응을 받은 제품")
    most_negative_product: str = Field(description="가장 부정적인 반응을 받은 제품")
    sentiment_distribution_analysis: str = Field(
        description="감정 분포 차트에서 관찰된 제품 간 차이 분석",
        max_length=300
    )
    common_praises: List[str] = Field(
        description="여러 제품에 공통적인 칭찬사항 (워드클라우드 기반)",
        min_length=2, max_length=5
    )
    common_complaints: List[str] = Field(
        description="여러 제품에 공통적인 불만사항 (워드클라우드 기반)",
        min_length=2, max_length=5
    )


# 키워드 연관성 
class KeywordAssociationInsight(BaseModel):
    """동시출현 네트워크 기반 키워드 연관성 분석"""
    positive_keyword_clusters: List[str] = Field(
        description="긍정 키워드 동시출현에서 발견된 주요 키워드 군집/패턴 (예: 발림성-촉촉함-보습 연결)",
        min_length=1, max_length=5
    )
    negative_keyword_clusters: List[str] = Field(
        description="부정 키워드 동시출현에서 발견된 주요 키워드 군집/패턴",
        min_length=1, max_length=5
    )
    cross_product_patterns: str = Field(
        description="제품 간 키워드 연관성 비교에서 발견된 패턴",
        max_length=300
    )


# 트렌드 & 제안
class KeyTrend(BaseModel):
    """주요 트렌드 및 패턴"""
    trend_name: str = Field(description="트렌드 이름")
    description: str = Field(description="트렌드 설명", max_length=200)
    affected_products: List[str] = Field(description="영향받는 제품들")


class ActionableRecommendation(BaseModel):
    """실행 가능한 개선 제안"""
    target_product: str = Field(description="대상 제품")
    category: Literal[
        "품질", "디자인", "서비스", "마케팅", "가격", "포장", "기능"
    ] = Field(description="개선 카테고리")
    recommendation: str = Field(description="구체적인 개선 제안", max_length=200)
    data_evidence: str = Field(
        description="제안의 근거가 되는 데이터/차트 (예: 레이더 차트에서 가성비 긍정 79%로 최하위)",
        max_length=150
    )
    expected_impact: str = Field(description="예상 효과", max_length=100)
    priority: Literal["높음", "중간", "낮음"] = Field(
        description="우선순위. 높음: 즉시 개선 필요. 중간: 중요하지만 시급하지 않음. 낮음: 장기적 과제"
    )


# 메인 스키마 
class ComprehensiveInsight(BaseModel):
    """종합 인사이트 분석 결과 (텍스트 + 시각화 멀티모달)"""
    executive_summary: str = Field(
        description="전체 제품 비교 분석 요약 (핵심 발견사항, 데이터 수치, 권장사항 포함)",
        max_length=600
    )
    chart_interpretations: List[ChartInterpretation] = Field(
        description="각 시각화 차트에 대한 해석 (첨부된 이미지 수만큼)",
        min_length=3, max_length=10
    )
    product_rankings: List[ProductRanking] = Field(
        description="제품별 종합 순위 및 평가",
        min_length=1
    )
    aspect_insights: List[AspectInsight] = Field(
        description="측면별 상세 인사이트 (품질, 사용성, 가성비, 기능성, 전반적은 필수)",
        min_length=5, max_length=10
    )
    sentiment_insights: SentimentInsight = Field(
        description="감정 분석 인사이트"
    )
    keyword_association: KeywordAssociationInsight = Field(
        description="키워드 동시출현 연관성 분석"
    )
    key_trends: List[KeyTrend] = Field(
        description="주요 트렌드 및 패턴",
        min_length=3, max_length=5
    )
    actionable_recommendations: List[ActionableRecommendation] = Field(
        description="실행 가능한 개선 제안 (우선순위별 정렬, 데이터 근거 필수)",
        min_length=3, max_length=10
    )
    market_positioning: str = Field(
        description="시장 포지셔닝 분석 (각 제품의 시장 내 위치와 타겟 고객)",
        max_length=500
    )
    competitive_advantages: str = Field(
        description="경쟁 우위 요소 분석 (제품별 차별화 포인트)",
        max_length=500
    )
    risk_factors: List[str] = Field(
        description="주의해야 할 위험 요소",
        min_length=2, max_length=5
    )


print("종합 인사이트 스키마 정의 완료")

### 2. 데이터 로드

**실습 (2)**
- 실전_1_2에서 저장한 텍스트 요약 파일과 시각화 이미지를 로드합니다.
- 이미지는 `BinaryContent`로 변환하여 PydanticAI Agent에 전달합니다.

In [ ]:
# =============================================================================
#  텍스트 + 이미지 데이터 로드
# =============================================================================

if not ANALYSIS_DIR.exists():
    print(f"[오류] {ANALYSIS_DIR} 폴더가 없습니다.")
    print("실전_1_2_제품비교_리뷰분석.ipynb 를 먼저 실행해주세요.")
else:
    # 1. 텍스트 요약 로드
    summary_file = ANALYSIS_DIR / "product_comparison_summary.txt"
    if summary_file.exists():
        summary_text = summary_file.read_text(encoding='utf-8')
        print(f"텍스트 요약 로드 완료: {len(summary_text)}자")
    else:
        summary_text = ""
        print("[경고] product_comparison_summary.txt 파일이 없습니다.")

    # 2. 이미지 로드 (BinaryContent로 변환)
    image_extensions = {'.png', '.jpg', '.jpeg', '.webp'}
    image_files = sorted([
        f for f in ANALYSIS_DIR.iterdir()
        if f.suffix.lower() in image_extensions
    ])

    image_contents = []
    for img_path in image_files:
        media_type = 'image/png' if img_path.suffix.lower() == '.png' else 'image/jpeg'
        with open(img_path, 'rb') as f:
            image_contents.append(
                BinaryContent(data=f.read(), media_type=media_type)
            )
        print(f"  이미지 로드: {img_path.name}")

    print()
    print(f"로드 완료:")
    print(f"  텍스트: {len(summary_text)}자")
    print(f"  이미지: {len(image_contents)}개")

### 3. 종합 인사이트 생성

**실습 (3)**
- 시스템 프롬프트에는 스키마 `description`과 중복되지 않는 **행동 지시만** 작성합니다.
- `BinaryContent` 이미지와 텍스트를 리스트로 묶어 Agent에 전달합니다.

In [ ]:
# =============================================================================
#  시스템 프롬프트 정의 + Agent 생성
# =============================================================================

insight_system_prompt = """
당신은 제품 비교 분석 전문가입니다.
제품 리뷰 데이터와 시각화 자료를 분석하여 실행 가능한 비즈니스 인사이트를 도출하세요.

[분석 원칙]
- 제공된 텍스트 요약과 시각화 이미지를 모두 활용하세요.
- 구체적인 수치와 근거를 제시하고, 추측보다 데이터에서 확인된 사실을 중심으로 분석하세요.
- 장점과 단점을 공정하게 평가하고, 각 제품의 고유한 가치를 인정하세요.
- 비즈니스 의사결정에 즉시 활용 가능한 구체적이고 실행 가능한 제안을 하세요.
- '~인 것 같다' 대신 '~로 나타났다' 등 데이터 기반 표현을 사용하세요.
"""

insight_agent = Agent(
    model_id,
    output_type=ComprehensiveInsight,
    system_prompt=insight_system_prompt,
)

print("Agent 생성 완료")
print(f"  모델: {model_id}")
print(f"  출력 스키마: ComprehensiveInsight")

In [ ]:
# =============================================================================
#  종합 인사이트 생성 실행
# =============================================================================

prompt_text = f"""다음은 여러 제품에 대한 리뷰 비교 분석 결과입니다.

## 텍스트 요약 데이터:
{summary_text}

## 첨부된 시각화 차트 목록:
아래 7종의 차트 이미지가 첨부되어 있습니다. 각 차트를 꼼꼼히 분석해주세요.

1. **평점 비교 차트** - 제품별 평균 평점(막대)과 추천도(라인) 비교
2. **감정 분포 차트** - 제품별 긍정/혼합/중립/부정 리뷰 건수 분포
3. **히트맵 차트** - 측면별(품질, 사용성, 가성비 등) 긍정 비율 히트맵
4. **레이더 차트** - 제품별 측면별 긍정/부정 비율 레이더 비교
5. **워드클라우드 차트** - 제품별 장점/단점 핵심 키워드 워드클라우드
6. **측면별 키워드 차트** - 제품별 측면별 긍정/부정 키워드 분포
7. **동시출현 차트** - 긍정/부정 키워드 간 동시출현 빈도 막대그래프

## 분석 요청:
chart_interpretations에는 위 7종 차트 각각에 대해 반드시 해석을 작성해주세요.
chart_name은 정확히 다음 이름을 사용하세요:
- "평점 비교 차트"
- "감정 분포 차트"
- "히트맵 차트"
- "레이더 차트"
- "워드클라우드 차트"
- "측면별 키워드 차트"
- "동시출현 차트"

각 차트에서 보이는 수치, 패턴, 분포를 구체적으로 언급하고,
그것이 소비자/비즈니스에 어떤 의미인지 시사점을 도출하세요.
텍스트 데이터와 시각화를 교차 분석하여 종합적인 인사이트를 작성해주세요."""

print("종합 인사이트 생성 중... (멀티모달 분석)")
print(f"  텍스트: {len(summary_text)}자")
print(f"  이미지: {len(image_contents)}개")
print()

# Agent 실행 (이미지 + 텍스트를 리스트로 전달)
result = await insight_agent.run(
    [*image_contents, prompt_text],
    model_settings=GoogleModelSettings(
        temperature=0.3,
        google_thinking_config={'thinking_level': 'high'},
    ),
)

insight = result.output
print("종합 인사이트 생성 완료!")

### 4. 결과 확인

**실습 (4)**
- 생성된 종합 인사이트의 주요 항목을 확인합니다.

In [ ]:
# =============================================================================
#  인사이트 결과 출력
# =============================================================================

print("=" * 80)
print("종합 인사이트 분석 결과 (텍스트 + 시각화 멀티모달)")
print("=" * 80)

# 1. 경영진 요약
print("\n[경영진 요약]")
print(insight.executive_summary)

# 2. 시각화 차트 해석
print("\n[시각화 차트 해석]")
for ci in insight.chart_interpretations:
    print(f"  ■ {ci.chart_name}")
    print(f"    관찰: {ci.data_observation}")
    print(f"    시사점: {ci.business_implication}")
    print()

# 3. 제품 순위
print("[제품 순위]")
for ranking in insight.product_rankings:
    print(f"  {ranking.rank}위: {ranking.product_name} ({ranking.overall_score})")
    print(f"    강점: {chr(10).join(f"      - {s}" for s in ranking.strengths)}")
    print(f"    약점: {chr(10).join(f"      - {w}" for w in ranking.weaknesses)}")
    print(f"    핵심 키워드: {', '.join(ranking.key_keywords)}")
    print(f"    추천 대상: {ranking.recommended_for}")
    print(f"    비추천 대상: {ranking.not_recommended_for}")
    print()

# 4. 측면별 인사이트
print("[측면별 인사이트]")
for asp in insight.aspect_insights:
    print(f"  {asp.aspect}: 최고={asp.best_product} / 최저={asp.worst_product}")
    print(f"    긍정비율: {asp.positive_ratio_comparison}")
    print(f"    => {asp.key_finding}")

# 5. 감정 분석
print(f"\n[감정 분석]")
print(f"  가장 긍정적: {insight.sentiment_insights.most_positive_product}")
print(f"  가장 부정적: {insight.sentiment_insights.most_negative_product}")
print(f"  감정 분포 분석: {insight.sentiment_insights.sentiment_distribution_analysis}")
print(f"  공통 칭찬: {', '.join(insight.sentiment_insights.common_praises)}")
print(f"  공통 불만: {', '.join(insight.sentiment_insights.common_complaints)}")

# 6. 키워드 연관성
print(f"\n[키워드 연관성 분석]")
ka = insight.keyword_association
print(f"  긍정 키워드 군집:")
for c in ka.positive_keyword_clusters:
    print(f"    - {c}")
print(f"  부정 키워드 군집:")
for c in ka.negative_keyword_clusters:
    print(f"    - {c}")
print(f"  제품 간 패턴: {ka.cross_product_patterns}")

# 7. 개선 제안
print(f"\n[개선 제안]")
for rec in insight.actionable_recommendations:
    print(f"  [{rec.priority}] {rec.target_product} - {rec.category}")
    print(f"    제안: {rec.recommendation}")
    print(f"    근거: {rec.data_evidence}")
    print(f"    효과: {rec.expected_impact}")
    print()

# 8. 시장 분석
print("[시장 포지셔닝]")
print(insight.market_positioning)
print()
print("[경쟁 우위]")
print(insight.competitive_advantages)
print()
print("[위험 요소]")
for risk in insight.risk_factors:
    print(f"  - {risk}")

### 5. 결과 저장

In [ ]:
# =============================================================================
#  결과 저장 (JSON + 마크다운 리포트)
# =============================================================================

ANALYSIS_DIR.mkdir(exist_ok=True)

# 1. JSON 저장
json_path = ANALYSIS_DIR / "comprehensive_insight.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(insight.model_dump(), f, ensure_ascii=False, indent=2)
print(f"JSON 저장: {json_path}")

# 2. 마크다운 리포트 생성
md_path = ANALYSIS_DIR / "comprehensive_insight_report.md"

# 이미지 파일 목록 수집
import glob as _glob
image_files = sorted(_glob.glob(str(ANALYSIS_DIR / "*.png")))
image_map = {}
for img in image_files:
    fname = os.path.basename(img)
    if "rating_recommendation" in fname:
        image_map["rating"] = fname
    elif "sentiment_distribution" in fname:
        image_map["sentiment"] = fname
    elif "aspect_positive_heatmap" in fname:
        image_map["heatmap"] = fname
    elif "product_radar" in fname:
        image_map["radar"] = fname
    elif "wordcloud" in fname:
        image_map.setdefault("wordclouds", []).append(fname)
    elif "cooccurrence" in fname and fname.endswith(".png"):
        image_map.setdefault("cooccurrence", []).append(fname)
    elif "aspect_keywords" in fname:
        image_map.setdefault("aspect_keywords", []).append(fname)

# 차트 해석을 이름으로 빠르게 찾기 위한 딕셔너리
chart_interp_map = {ci.chart_name: ci for ci in insight.chart_interpretations}


def get_chart_analysis(chart_name):
    """차트 이름으로 해석 찾기"""
    ci = chart_interp_map.get(chart_name)
    if ci:
        return f"\n> **데이터 관찰:** {ci.data_observation}\n>\n> **비즈니스 시사점:** {ci.business_implication}\n"
    # 부분 매칭 fallback
    for name, ci in chart_interp_map.items():
        if chart_name[:2] in name:
            return f"\n> **데이터 관찰:** {ci.data_observation}\n>\n> **비즈니스 시사점:** {ci.business_implication}\n"
    return ""


lines = []

# ── 표지 ──
lines.append("# 쿠팡 선크림 제품 비교 분석 종합 리포트\n")
lines.append(f"> **분석 대상:** 쿠팡 선크림 리뷰 데이터  ")
lines.append(f"> **분석 제품:** {len(insight.product_rankings)}개 제품 비교  ")
lines.append(f"> **분석 방법:** 텍스트 통계 + 시각화 차트 멀티모달 AI 분석\n")
lines.append("---\n")

# ── 목차 ──
lines.append("## 목차\n")
lines.append("1. [경영진 요약](#1-경영진-요약)")
lines.append("2. [시각화 분석](#2-시각화-분석)")
lines.append("3. [제품 순위](#3-제품-순위)")
lines.append("4. [측면별 인사이트](#4-측면별-인사이트)")
lines.append("5. [감정 분석](#5-감정-분석)")
lines.append("6. [키워드 연관성 분석](#6-키워드-연관성-분석)")
lines.append("7. [주요 트렌드](#7-주요-트렌드)")
lines.append("8. [개선 제안](#8-개선-제안)")
lines.append("9. [시장 분석](#9-시장-분석)")
lines.append("10. [위험 요소](#10-위험-요소)\n")
lines.append("---\n")

# ── 1. 경영진 요약 ──
lines.append("## 1. 경영진 요약\n")
lines.append(f"{insight.executive_summary}\n")

# ── 2. 시각화 분석 (이미지 + AI 해석) ──
lines.append("## 2. 시각화 분석\n")

if "rating" in image_map:
    lines.append("### 2-1. 평균 평점 및 추천도 비교\n")
    lines.append(f"![평균 평점 및 추천도 비교]({image_map['rating']})\n")
    lines.append(get_chart_analysis("평점 비교 차트"))

if "sentiment" in image_map:
    lines.append("### 2-2. 감정 분포 비교\n")
    lines.append(f"![감정 분포 비교]({image_map['sentiment']})\n")
    lines.append(get_chart_analysis("감정 분포 차트"))

if "heatmap" in image_map:
    lines.append("### 2-3. 항목별 긍정 비율 히트맵\n")
    lines.append(f"![항목별 긍정 비율 히트맵]({image_map['heatmap']})\n")
    lines.append(get_chart_analysis("히트맵 차트"))

if "radar" in image_map:
    lines.append("### 2-4. 제품별 레이더 차트\n")
    lines.append(f"![제품별 레이더 차트]({image_map['radar']})\n")
    lines.append(get_chart_analysis("레이더 차트"))

if "wordclouds" in image_map:
    lines.append("### 2-5. 장단점 워드클라우드\n")
    for wc in image_map["wordclouds"]:
        lines.append(f"![워드클라우드]({wc})\n")
    lines.append(get_chart_analysis("워드클라우드 차트"))

if "aspect_keywords" in image_map:
    lines.append("### 2-6. 측면별 키워드 분석\n")
    for ak in image_map["aspect_keywords"]:
        lines.append(f"![측면별 키워드]({ak})\n")
    lines.append(get_chart_analysis("측면별 키워드 차트"))

if "cooccurrence" in image_map:
    lines.append("### 2-7. 키워드 동시출현 분석\n")
    for co in image_map["cooccurrence"]:
        lines.append(f"![동시출현 분석]({co})\n")
    lines.append(get_chart_analysis("동시출현 차트"))

# ── 3. 제품 순위 ──
lines.append("## 3. 제품 순위\n")
for r in insight.product_rankings:
    lines.append(f"### {r.rank}위: {r.product_name}\n")
    lines.append(f"**종합 점수:** {r.overall_score}\n")
    lines.append("**강점:**")
    for s in r.strengths:
        lines.append(f"- {s}")
    lines.append("\n**약점:**")
    for w in r.weaknesses:
        lines.append(f"- {w}")
    lines.append(f'\n**핵심 키워드:** {chr(44).join(f" `{k}`" for k in r.key_keywords)}')
    lines.append(f"\n**추천 대상:** {r.recommended_for}\n")
    lines.append(f"**비추천 대상:** {r.not_recommended_for}\n")

# ── 4. 측면별 인사이트 ──
lines.append("## 4. 측면별 인사이트\n")
lines.append("| 측면 | 최고 제품 | 최저 제품 | 긍정 비율 비교 |")
lines.append("|------|----------|----------|----------------|")
for a in insight.aspect_insights:
    lines.append(f"| {a.aspect} | {a.best_product} | {a.worst_product} | {a.positive_ratio_comparison} |")
lines.append("")
for a in insight.aspect_insights:
    lines.append(f"- **{a.aspect}**: {a.key_finding}")
lines.append("")

# ── 5. 감정 분석 ──
lines.append("## 5. 감정 분석\n")
si = insight.sentiment_insights
lines.append(f"- **가장 긍정적:** {si.most_positive_product}")
lines.append(f"- **가장 부정적:** {si.most_negative_product}\n")
lines.append(f"{si.sentiment_distribution_analysis}\n")
lines.append(f'**공통 칭찬:** {chr(44).join(f" `{p}`" for p in si.common_praises)}\n')
lines.append(f'**공통 불만:** {chr(44).join(f" `{c}`" for c in si.common_complaints)}\n')

# ── 6. 키워드 연관성 분석 ──
lines.append("## 6. 키워드 연관성 분석\n")
ka = insight.keyword_association
lines.append("### 긍정 키워드 군집\n")
for c in ka.positive_keyword_clusters:
    lines.append(f"- {c}")
lines.append("\n### 부정 키워드 군집\n")
for c in ka.negative_keyword_clusters:
    lines.append(f"- {c}")
lines.append(f"\n### 제품 간 패턴\n")
lines.append(f"{ka.cross_product_patterns}\n")

# ── 7. 주요 트렌드 ──
lines.append("## 7. 주요 트렌드\n")
for t in insight.key_trends:
    lines.append(f"### {t.trend_name}\n")
    lines.append(f"{t.description}\n")
    lines.append(f'- 영향 제품: {chr(44).join(f" {p}" for p in t.affected_products)}\n')

# ── 8. 개선 제안 ──
lines.append("## 8. 개선 제안\n")
lines.append("| 우선순위 | 제품 | 카테고리 | 제안 | 근거 | 예상 효과 |")
lines.append("|---------|------|---------|------|------|----------|")
for rec in insight.actionable_recommendations:
    lines.append(f"| {rec.priority} | {rec.target_product} | {rec.category} | {rec.recommendation} | {rec.data_evidence} | {rec.expected_impact} |")
lines.append("")

# ── 9. 시장 분석 ──
lines.append("## 9. 시장 분석\n")
lines.append("### 포지셔닝\n")
lines.append(f"{insight.market_positioning}\n")
lines.append("### 경쟁 우위\n")
lines.append(f"{insight.competitive_advantages}\n")

# ── 10. 위험 요소 ──
lines.append("## 10. 위험 요소\n")
for risk in insight.risk_factors:
    lines.append(f"- {risk}")
lines.append("")

# ── 푸터 ──
lines.append("---\n")
lines.append("*본 리포트는 쿠팡 리뷰 데이터 기반의 AI 분석이며,")
lines.append("텍스트 통계와 시각화 차트를 멀티모달로 종합 분석한 결과입니다.*")

md_content = "\n".join(lines)
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_content)
print(f"마크다운 리포트 저장: {md_path}")

print()
print("전체 프로세스 완료!")
print(f"  {json_path}")
print(f"  {md_path}")

### 6. 결과 확인

생성된 파일:
- `outputs/coupang_product_comparison/comprehensive_insight.json` - JSON 형식 원본 데이터
- `outputs/coupang_product_comparison/comprehensive_insight_report.md` - 마크다운 리포트

주요 인사이트:
- 제품별 순위 및 강점/약점
- 측면별 최고/최저 제품
- 감정 분석 (긍정/부정 패턴)
- 주요 트렌드
- 실행 가능한 개선 제안 (우선순위별)
- 시장 분석 및 위험 요소